In [2]:
# Cellule 1 : Vérification du GPU disponible sur Colab
!nvidia-smi

# Cellule 2 : Installation et imports
import time
import math
import numpy as np
from numba import cuda
from numba.cuda.random import create_xoroshiro128p_states, xoroshiro128p_normal_float64

# Cellule 3 : Lancement sur VRAI GPU à grande échelle (N = 100 Millions)
N_TOTAL = 100_000_000
threads_per_block = 256
blocks = 1024
total_threads = blocks * threads_per_block
N_per_thread = math.ceil(N_TOTAL / total_threads)

print(f"Lancement de {N_TOTAL:,} simulations sur GPU réel...")

# Mesure du temps
start_time = time.time()

rng_states = create_xoroshiro128p_states(total_threads, seed=1234)
d_payoffs = cuda.device_array(total_threads, dtype=np.float64)

# Exécution du noyau GPU
monte_carlo_option_kernel[blocks, threads_per_block](
    rng_states, 100.0, 100.0, 1.0, 0.05, 0.20, N_per_thread, d_payoffs
)
cuda.synchronize()

gpu_time = time.time() - start_time
final_price = float(np.mean(d_payoffs.copy_to_host()))

print(f"Prix calculé : {final_price:.4f} €")
print(f"Temps GPU total : {gpu_time:.4f} secondes")
print(f"Débit : {N_TOTAL / gpu_time / 1e6:.2f} Millions de simulations / seconde")

zsh:1: command not found: nvidia-smi


ModuleNotFoundError: No module named 'numba'